# PaMCP E0 — Registry characterization (audited campaign)
This notebook executes E0 with label-coverage checks, immutable timestamped output, and explicit silver-label caveats. It does not execute E1/E2 or call external models.

In [ ]:
from pathlib import Path
import subprocess, sys, datetime
repo = Path('/content/taas-provenance-discovery')
if not repo.exists():
    subprocess.run(['git','clone','--branch','research/pamcp-multibenchmark-v1','https://github.com/msellamiTN/taas-provenance-discovery.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'fetch','origin','research/pamcp-multibenchmark-v1'], check=True)
    subprocess.run(['git','-C',str(repo),'checkout','research/pamcp-multibenchmark-v1'], check=True)
    subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin','research/pamcp-multibenchmark-v1'], check=True)
print('HEAD:', subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip())

In [ ]:
import os
os.chdir(repo)
stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out = repo / 'results' / 'campaigns' / f'e0-registry-{stamp}.json'
env = os.environ.copy(); env['PYTHONPATH'] = str(repo / 'src')
cmd = [sys.executable, '-m', 'experiments.e0_registry_campaign', '--root', str(repo), '--output', str(out)]
p = subprocess.run(cmd, env=env, text=True, capture_output=True)
print(p.stdout)
if p.stderr: print(p.stderr)
if p.returncode: raise RuntimeError(f'E0 failed with exit code {p.returncode}')
print('Immutable report:', out)

In [ ]:
import json
report = json.loads(out.read_text())
print(json.dumps({
 'status': report['status'],
 'coverage': report['coverage'],
 'published_annotations': report['published_annotations'],
 'silver_label_analysis': report['silver_label_analysis'],
 'limitations': report['limitations'],
}, indent=2))


## Interpretation rules
- `lexical_vs_silver` measures agreement with the repository's model-generated silver labels only.
- Missing silver labels are excluded from comparison metrics and explicitly counted.
- Published `readOnlyHint` annotations are metadata claims, not observed effects.
- The JSON report is created with exclusive mode; an existing report is never overwritten.
- Do not present these numbers as independently adjudicated accuracy.